In [1]:
import pickle
from pathlib import Path

import pandas as pd
import numpy as np

DATA_DIR = Path("../data/news-portal-user-interactions-by-globocom")
CLICKS_DIR = DATA_DIR / "clicks" / "clicks"

articles_metadata_path = DATA_DIR / "articles_metadata.csv"
clicks_sample_path = DATA_DIR / "clicks_sample.csv"
embeddings_path = DATA_DIR / "articles_embeddings.pickle"

# Chargement
articles_metadata = pd.read_csv(articles_metadata_path)
clicks_sample = pd.read_csv(clicks_sample_path)

with open(embeddings_path, "rb") as f:
    embeddings = pickle.load(f)

print("articles_metadata shape:", articles_metadata.shape)
print("clicks_sample shape:", clicks_sample.shape)
print("type embeddings:", type(embeddings))

/tmp/ipykernel_31851/2722869837.py:19: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  embeddings = pickle.load(f)


articles_metadata shape: (364047, 5)
clicks_sample shape: (1883, 12)
type embeddings: <class 'numpy.ndarray'>


In [2]:
articles_metadata.head()

,article_id,category_id,created_at_ts,publisher_id,words_count
0,0,0,1513144419000,0,168
1,1,1,1405341936000,0,189
2,2,1,1408667706000,0,250
3,3,1,1408468313000,0,230
4,4,1,1407071171000,0,162


In [3]:
clicks_sample.head()

,user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,1506825423271737,1506825423000,2,157541,1506826828020,4,3,20,1,20,2
1,0,1506825423271737,1506825423000,2,68866,1506826858020,4,3,20,1,20,2
2,1,1506825426267738,1506825426000,2,235840,1506827017951,4,1,17,1,16,2
3,1,1506825426267738,1506825426000,2,96663,1506827047951,4,1,17,1,16,2
4,2,1506825435299739,1506825435000,2,119592,1506827090575,4,1,17,1,24,2


In [4]:
articles_metadata.columns.tolist(), clicks_sample.columns.tolist()

(['article_id', 'category_id', 'created_at_ts', 'publisher_id', 'words_count'],
 ['user_id',
  'session_id',
  'session_start',
  'session_size',
  'click_article_id',
  'click_timestamp',
  'click_environment',
  'click_deviceGroup',
  'click_os',
  'click_country',
  'click_region',
  'click_referrer_type'])

In [5]:
articles_metadata.dtypes

article_id       int64
category_id      int64
created_at_ts    int64
publisher_id     int64
words_count      int64
dtype: object

In [6]:
articles_metadata.isna().sum().sort_values(ascending=False)

article_id       0
category_id      0
created_at_ts    0
publisher_id     0
words_count      0
dtype: int64

In [7]:
clicks_sample.isna().sum().sort_values(ascending=False)

user_id                0
session_id             0
session_start          0
session_size           0
click_article_id       0
click_timestamp        0
click_environment      0
click_deviceGroup      0
click_os               0
click_country          0
click_region           0
click_referrer_type    0
dtype: int64

In [8]:
# Inspection embeddings
if isinstance(embeddings, dict):
    print("Nombre d'articles dans embeddings :", len(embeddings))
    first_key = next(iter(embeddings))
    print("Premier article_id :", first_key)
    print("Type vecteur :", type(embeddings[first_key]))
    print("Longueur vecteur :", len(embeddings[first_key]))
elif isinstance(embeddings, pd.DataFrame):
    print(embeddings.shape)
    display(embeddings.head())
elif isinstance(embeddings, np.ndarray):
    print(embeddings.shape)
else:
    print("Type non prévu :", type(embeddings))

(364047, 250)


In [9]:
articles_metadata["article_id"].head(10).tolist(), articles_metadata["article_id"].tail(10).tolist()

([0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [364037,
  364038,
  364039,
  364040,
  364041,
  364042,
  364043,
  364044,
  364045,
  364046])

In [10]:
articles_metadata["article_id"].min(), articles_metadata["article_id"].max(), articles_metadata["article_id"].nunique()

(np.int64(0), np.int64(364046), 364047)

In [11]:
(
    articles_metadata["article_id"].sort_values().reset_index(drop=True)
    == pd.Series(range(len(articles_metadata)))
).all()

np.True_

In [12]:
sample_article_ids = set(clicks_sample["click_article_id"].unique())
metadata_article_ids = set(articles_metadata["article_id"].unique())

missing_in_metadata = sample_article_ids - metadata_article_ids
len(missing_in_metadata), list(sorted(missing_in_metadata))[:10]

(0, [])

In [13]:
max_article_id = articles_metadata["article_id"].max()
invalid_for_embeddings = clicks_sample.loc[
    (clicks_sample["click_article_id"] < 0) |
    (clicks_sample["click_article_id"] > max_article_id),
    "click_article_id"
].unique()

len(invalid_for_embeddings), invalid_for_embeddings[:10]

(0, array([], dtype=int64))

In [14]:
clicks_sample["user_id"].nunique(), clicks_sample["click_article_id"].nunique()

(707, 323)

In [15]:
clicks_sample.groupby("user_id")["click_article_id"].count().sort_values(ascending=False).head(10)

user_id
681    24
64     16
183    10
283     9
255     9
157     8
102     8
286     8
16      7
244     7
Name: click_article_id, dtype: int64